# optimizer-state-tensor-buffers — faded example 3: In-place buffer update via copy_ inside step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-state-tensor-buffers`. Running the beacon reports progress on the `Optimizer: Per-param state buffers` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Per-param state buffers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-state-tensor-buffers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-state-tensor-buffers"
DD_SUBTOPIC = "Optimizer: Per-param state buffers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To accumulate momentum correctly, the velocity buffer must be updated in-place using `buf.copy_(new_value)`. An out-of-place assignment `buf = new_value` creates a new tensor and breaks the reference held in `self.bufs`, so subsequent steps see a stale all-zero buffer instead of the accumulated velocity.

## Faded exercise 3

The `__init__` allocates `self.bufs` correctly. Complete the loop body in `step`: update each buffer in-place using `b.copy_(self.momentum * b + p.grad)`, then apply the update to the parameter.

**Fill in:** The two in-place lines: b.copy_(self.momentum * b + p.grad) to update the buffer, then p -= self.lr * b to update the parameter.

In [ ]:
import torch as t
import torch.nn as nn

class MomentumSGD2:
    def __init__(self, params, lr, momentum=0.9):
        self.params   = list(params)
        self.lr       = lr
        self.momentum = momentum
        self.bufs     = [t.zeros_like(p) for p in self.params]

    @t.inference_mode()
    def step(self):
        for p, b in zip(self.params, self.bufs):
            if p.grad is None:
                continue
            b.copy_(self.momentum * b + p.grad)
            p -= self.lr * b

    def zero_grad(self):
        for p in self.params: p.grad = None

# --- run it ---
t.manual_seed(0)
model = nn.Linear(4, 2)
opt = MomentumSGD2(model.parameters(), lr=0.01, momentum=0.9)
loss = model(t.randn(5, 4)).sum()
loss.backward()
buf_ptr_before = opt.bufs[0].data_ptr()
opt.step()
buf_ptr_after = opt.bufs[0].data_ptr()
print(f'buf ptr unchanged: {buf_ptr_before == buf_ptr_after}')  # True


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(0)
    model_a = nn.Linear(4, 2)
    model_b = nn.Linear(4, 2)
    for pa, pb in zip(model_a.parameters(), model_b.parameters()):
        pb.data.copy_(pa.data)
    opt_ours  = MomentumSGD2(model_a.parameters(), lr=0.01, momentum=0.9)
    opt_torch = t.optim.SGD(model_b.parameters(), lr=0.01, momentum=0.9)
    data = t.randn(5, 4, generator=t.Generator().manual_seed(11))
    ptrs_before = [b.data_ptr() for b in opt_ours.bufs]
    for _ in range(5):
        out_a = model_a(data).sum(); out_a.backward()
        opt_ours.step(); opt_ours.zero_grad()
        out_b = model_b(data).sum(); out_b.backward()
        opt_torch.step(); opt_torch.zero_grad()
    ptrs_after = [b.data_ptr() for b in opt_ours.bufs]
    assert ptrs_before == ptrs_after, 'Buffer data pointers must be stable (in-place update)'
    for pa, pb in zip(model_a.parameters(), model_b.parameters()):
        assert t.allclose(pa, pb, atol=1e-5), f'Mismatch: {pa} vs {pb}'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class MomentumSGD2:
    def __init__(self, params, lr, momentum=0.9):
        self.params   = list(params)
        self.lr       = lr
        self.momentum = momentum
        self.bufs     = [t.zeros_like(p) for p in self.params]

    @t.inference_mode()
    def step(self):
        for p, b in zip(self.params, self.bufs):
            if p.grad is None:
                continue
            b.copy_(self.momentum * b + p.grad)
            p -= self.lr * b

    def zero_grad(self):
        for p in self.params: p.grad = None

# --- run it ---
t.manual_seed(0)
model = nn.Linear(4, 2)
opt = MomentumSGD2(model.parameters(), lr=0.01, momentum=0.9)
loss = model(t.randn(5, 4)).sum()
loss.backward()
buf_ptr_before = opt.bufs[0].data_ptr()
opt.step()
buf_ptr_after = opt.bufs[0].data_ptr()
print(f'buf ptr unchanged: {buf_ptr_before == buf_ptr_after}')  # True
```
</details>